# 1. Import Data

In [1]:
# Standard libs
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional, Tuple, List
import os
import glob
import json
from typing import Optional
from typing import Optional, Dict, Any
from itertools import product
from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import List, Optional, Dict, Any, Tuple

# Utils
import math
import random
import gc

# Data & Processing
import numpy as np
import pandas as pd
import geopandas as gpd

# Scikit-Learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, confusion_matrix
from sklearn.neighbors import BallTree, NearestNeighbors
from sklearn.model_selection import train_test_split

# TQDM
from tqdm.notebook import tqdm
from joblib import Parallel, delayed

# Plotting
import matplotlib.pyplot as plt

# PyTorch
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

# PyTorch Geometric
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GatedGraphConv, GATConv, ChebConv


# Confirm the current working directory
print("Current working directory:", os.getcwd())

Current working directory: /home/qusta100/STGNN


In [2]:
# Load data
df = pd.read_csv(
    "/gpfs/scratch/qusta100/STGNN/Data/Temp/final.csv",
    dtype={9: str}
)

In [3]:
# Filter for one specific day
# Because of limited computational resources, I restrict the dataset to a single day, which gives me 96 observations.
#df = df[df['date'].str.startswith("2025-04-01")]
df["date"] = pd.to_datetime(df["date"])
df.head()

,date,station_uuid,diesel,e5,e10,uuid,name,brand,street,house_number,...,t_87,t_88,t_89,t_90,t_91,t_92,t_93,t_94,t_95,weekday
0,2025-04-01 00:00:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18.0,...,False,False,False,False,False,False,False,False,False,1
1,2025-04-01 00:15:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18.0,...,False,False,False,False,False,False,False,False,False,1
2,2025-04-01 00:30:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18.0,...,False,False,False,False,False,False,False,False,False,1
3,2025-04-01 00:45:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18.0,...,False,False,False,False,False,False,False,False,False,1
4,2025-04-01 01:00:00,00060065-7890-4444-8888-acdc00000004,1.559,1.709,1.649,00060065-7890-4444-8888-acdc00000004,Georg Ultsch GmbH,Tankstelle Lichtenfels,Robert-Koch-Str.,18.0,...,False,False,False,False,False,False,False,False,False,1


# 2. STGNN Design

In [5]:
# =============================
# a) Parameters
# =============================

STATION_COL = "station_uuid"
LAT_COL = "latitude"
LON_COL = "longitude"
TIME_COL = "date"

# Default-Targets
TARGET_COLS_DEFAULT = ["e5"]
FEATURE_COLS_DEFAULT = ["e5"]  # standard: Targets = Features

RADIUS_KM = 10
NEIGHBOUR = 5
SEED = 42

EMBED_DIM = 32
HIDDEN_DIM = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 100
PATIENCE = 10
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

WINDOW_SIZE = 672 # high value due to different encoder
HORIZON_STEPS = 4  # Horizont per Target
LENGTH_ENCODER_DEFAULT = 16 # Target Length after Strides


# =============================
# b) Helper functions
# =============================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def km_to_radians(km: float) -> float:
    earth_radius_km = 6371.0088
    return km / earth_radius_km


def build_knn_graph(
    lat_lon_deg: np.ndarray,
    k: int = 5,
    return_dists: bool = False,
    RADIUS_KM: Optional[float] = None
) -> np.ndarray | Tuple[np.ndarray, np.ndarray]:
    
    lat_lon_rad = np.radians(lat_lon_deg)
    tree = BallTree(lat_lon_rad, metric="haversine")

    dist_rad, ind = tree.query(lat_lon_rad, k=k + 1)

    src, dst, dists_km = [], [], []
    radius_rad = km_to_radians(RADIUS_KM) if RADIUS_KM is not None else None

    for i, (neigh_idx, neigh_dist_rad) in enumerate(zip(ind, dist_rad)):
        for j, dij_rad in zip(neigh_idx[1:], neigh_dist_rad[1:]):
            if radius_rad is not None and dij_rad > radius_rad:
                continue

            dij_km = dij_rad * 6371.0088  # Conversion for output
            src.append(j)
            dst.append(i)
            dists_km.append(dij_km)

    edge_index = np.vstack([np.array(src, dtype=int), np.array(dst, dtype=int)])

    if return_dists:
        return edge_index, np.asarray(dists_km, dtype=float)
    return edge_index


# =============================
# c) Window-based ST data (multi-step, multi-target)
# =============================

@dataclass
class STWindowDataMulti:
    x: torch.Tensor             # [T, N, F] full time series, F = num_features
    y: torch.Tensor             # [T, N, H * num_targets] H Horizonte für alle Targets
    edge_index: torch.Tensor
    edge_weight: torch.Tensor 
    train_end_idx: np.ndarray   # time indices where windows end (train)
    val_end_idx: np.ndarray     # time indices (validation)
    test_end_idx: np.ndarray    # time indices (test)
    valid_nodes: np.ndarray     # node indices (z.B. Thüringen)
    times: np.ndarray           # sorted timestamps
    meta: pd.DataFrame          # station metadata (uuid, lat, lon, node_id)
    feature_cols: List[str]     # relevant Feature-Spalten
    target_cols: List[str]      # relevant Target-Spalten
    time_feat: np.ndarray       # [T, time_in_dim], z.B. weekday, holiday, time_sin


def make_window_data_multi_from_df(
    df: pd.DataFrame,
    window_size: int = WINDOW_SIZE,
    horizon_steps: int = HORIZON_STEPS,
    feature_cols: Optional[List[str]] = None,
    target_cols: Optional[List[str]] = None,
    stride: int = 1, 
) -> STWindowDataMulti:

    if target_cols is None:
        target_cols = TARGET_COLS_DEFAULT
    if feature_cols is None:
        feature_cols = FEATURE_COLS_DEFAULT

    # Necessary Columns
    base_cols = [STATION_COL, LAT_COL, LON_COL, TIME_COL, "in_thuringia"]
    all_val_cols = sorted(set(feature_cols) | set(target_cols))
    needed = base_cols + all_val_cols
    assert all(c in df.columns for c in needed), f"Missing columns: {set(needed) - set(df.columns)}"

    df = df.copy()

    # Stations + node ids
    stations = df[[STATION_COL, LAT_COL, LON_COL]].drop_duplicates().reset_index(drop=True)
    stations["node_id"] = np.arange(len(stations))
    station2id = stations.set_index(STATION_COL)["node_id"].to_dict()

    # Time axis
    times = np.sort(df[TIME_COL].unique())
    time2id = {t: i for i, t in enumerate(times)}
    
    # ---- Build time features: weekday, holiday, time_sin ----
    
    weekday_cols = [f"wd_{i}" for i in range(7)]
    time_cols = [f"t_{i}" for i in range(96)]
    
    time_df = (
        df[[TIME_COL, *weekday_cols, "holiday", *time_cols]]
        .drop_duplicates(subset=[TIME_COL])
        .sort_values(TIME_COL)
    )

    assert np.array_equal(time_df[TIME_COL].to_numpy(), times), \
        "The timelines of time_df and times do not match."

    time_feat = time_df[[*weekday_cols, "holiday", *time_cols]].to_numpy(dtype=float)

    N = len(stations)
    T = len(times)
    M = len(target_cols)             # Number Targets
    F = len(feature_cols)            # Number Features
    H = horizon_steps

    # Spatial graph
    lat_lon = stations[[LAT_COL, LON_COL]].to_numpy(dtype=float)
    edge_index_np, edge_dist_km = build_knn_graph(
        lat_lon_deg=lat_lon,
        k=NEIGHBOUR,
        return_dists=True,
        RADIUS_KM=RADIUS_KM
    )
    edge_index = torch.tensor(edge_index_np, dtype=torch.long)
    
    # Distance -> Weight: the closer, the greater
    sigma = np.median(edge_dist_km)
    edge_weight_np = np.exp(-(edge_dist_km**2) / (2*sigma**2))
    edge_weight = torch.tensor(edge_weight_np, dtype=torch.float32)

    # Tensors
    x = torch.zeros((T, N, F), dtype=torch.float32)                    # [T, N, F]
    y = torch.full((T, N, H * M), float("nan"), dtype=torch.float32)   # [T, N, H*M]

    # Aggregation per (station, time) across all relevant columns
    df_idx = (
        df[[STATION_COL, TIME_COL] + all_val_cols]
        .groupby([STATION_COL, TIME_COL], as_index=False)
        .mean()
        .sort_values(TIME_COL)
    )

    df_idx["node_id"] = df_idx[STATION_COL].map(station2id)
    df_idx["time_id"] = df_idx[TIME_COL].map(time2id)

    # x[t, n, f] = Features at time t
    for _, row in df_idx.iterrows():
        t = int(row["time_id"])
        n = int(row["node_id"])
        for f_idx, col in enumerate(feature_cols):
            v = row[col]
            if pd.notna(v):
                x[t, n, f_idx] = float(v)

    # y[t, n, m*H + h] = Target m at time t+h+1
    for station, g in df_idx.groupby(STATION_COL):
        g = g.sort_values("time_id")
        node_id = int(g["node_id"].iloc[0])
        t_ids = g["time_id"].to_numpy()

        L = len(t_ids)
        for m, col in enumerate(target_cols):
            vals = g[col].to_numpy()

            for idx in range(L):
                t = int(t_ids[idx])
                if idx + H >= L:
                    break

                future_vals = vals[idx + 1: idx + 1 + H]

                if np.any(np.isnan(future_vals)):
                    continue

                for h in range(H):
                    y[t, node_id, m * H + h] = float(future_vals[h])

    # Time indices where full horizon is available
    effective_T = T - H
    if effective_T < window_size:
        raise ValueError(f"Too few timesteps ({effective_T}) for window size {window_size} and horizon {H}.")

    # Time-based split over 0..effective_T-1
    time_indices = np.arange(effective_T)
    num_total = len(time_indices)
    num_train = int((1.0 - VAL_SPLIT - TEST_SPLIT) * num_total)
    num_val   = int(VAL_SPLIT * num_total)
    num_test  = num_total - num_train - num_val

    train_t = time_indices[:num_train]
    val_t   = time_indices[num_train:num_train + num_val]
    test_t  = time_indices[num_train + num_val:]

    # Window end indices
    min_end = window_size - 1
    train_end_idx = train_t[train_t >= min_end]
    val_end_idx   = val_t[val_t >= min_end]
    test_end_idx  = test_t[test_t >= min_end]
    
    # stride
    train_end_idx = train_end_idx[::stride]
    val_end_idx   = val_end_idx[::stride]
    test_end_idx  = test_end_idx[::stride]

    # Thuringia mask
    th_mask = (
        df[[STATION_COL, "in_thuringia"]]
        .drop_duplicates()
        .set_index(STATION_COL)["in_thuringia"]
    )
    stations = stations.join(th_mask, on=STATION_COL)
    valid_nodes = stations.index[stations["in_thuringia"] == 1].to_numpy()

    if valid_nodes.size == 0:
        raise ValueError("No nodes with in_thuringia == 1 found.")

    meta = stations[[STATION_COL, LAT_COL, LON_COL, "node_id"]].copy()

    return STWindowDataMulti(
        x=x,
        y=y,
        edge_index=edge_index,
        edge_weight=edge_weight, 
        train_end_idx=train_end_idx,
        val_end_idx=val_end_idx,
        test_end_idx=test_end_idx,
        valid_nodes=valid_nodes,
        times=times,
        meta=meta,
        feature_cols=feature_cols,
        target_cols=target_cols,
        time_feat=time_feat,
    )


# -------------------------------------------------
# 1) Temporaler Downsampling-Encoder (Conv1d)
# -------------------------------------------------

class TemporalDownsamplingEncoder(nn.Module):
    def __init__(
        self,
        in_dim: int,
        mid_dim: int,
        out_dim: int,
        kernel1: int = 3,
        stride1: int = 1,
        kernel2: int = 3,
        stride2: int = 2,
    ):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels=in_dim,
            out_channels=mid_dim,
            kernel_size=kernel1,
            stride=stride1,
            padding=kernel1 // 2,
        )
        self.bn1 = nn.BatchNorm1d(mid_dim)

        self.conv2 = nn.Conv1d(
            in_channels=mid_dim,
            out_channels=out_dim,
            kernel_size=kernel2,
            stride=stride2,
            padding=0,
        )
        self.bn2 = nn.BatchNorm1d(out_dim)

        self.act = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [T, N, F]
        """
        T, N, F = x.shape

        x_ = x.permute(1, 2, 0)   # [N, F, T]

        x_ = self.conv1(x_)       # [N, mid_dim, T']
        x_ = self.bn1(x_)
        x_ = self.act(x_)

        x_ = self.conv2(x_)       # [N, out_dim, T_out]
        x_ = self.bn2(x_)
        x_ = self.act(x_)

        x_ = x_.permute(2, 0, 1)  # [T_out, N, out_dim]
        return x_


# -------------------------------------------------
# 2) Multi-Scale Temporaler Encoder
# -------------------------------------------------

def _downsample_stride(win: int, out_len: int) -> int:
    if win % out_len != 0:
        raise ValueError(f"win={win} must be divisible by out_len={out_len}.")
    return win // out_len


class MultiScaleTemporalEncoder(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim_per_scale_week: int,
        out_dim_per_scale_day: int,
        win_day: int,
        win_week: int,
        length_encoder: int = 16, 
    ):
        super().__init__()
        self.win_day = win_day
        self.win_week = win_week
        self.length_encoder = length_encoder
        
        stride_day = _downsample_stride(win_day, length_encoder)
        stride_week = _downsample_stride(win_week, length_encoder)


        self.enc_day = TemporalDownsamplingEncoder(
            in_dim=in_dim,
            mid_dim=hidden_dim,
            out_dim=out_dim_per_scale_day,
            kernel1=3,
            stride1=1,
            kernel2=stride_day,
            stride2=stride_day,
        )

        self.enc_week = TemporalDownsamplingEncoder(
            in_dim=in_dim,
            mid_dim=hidden_dim,
            out_dim=out_dim_per_scale_week,
            kernel1=3,
            stride1=1,
            kernel2=stride_week,
            stride2=stride_week,
        )

    @property
    def out_dim(self) -> int:
        return self.enc_day.conv2.out_channels + self.enc_week.conv2.out_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [W_global, N, F]
        """
        W_global, N, F = x.shape
        assert self.win_day  <= W_global
        assert self.win_week <= W_global

        x_day  = x[-self.win_day:]    # [Wd, N, F]
        x_week = x[-self.win_week:]   # [Ww, N, F]

        h_day  = self.enc_day(x_day)   # [Pd, N, C]
        h_week = self.enc_week(x_week) # [Pw, N, C]

        P = min(h_day.size(0), h_week.size(0))
        h_day  = h_day[-P:]
        h_week = h_week[-P:]

        h = torch.cat([h_day, h_week], dim=-1)  # [P, N, 2C]
        return h


# -------------------------------------------------
# 3) Local spatial convolution
# -------------------------------------------------

class LocalSpatialEncoder(nn.Module):
    def __init__(self, hidden_dim: int, cheb_K: int = 3):
        super().__init__()
        # Chebyshev-Conv über K-Hops
        self.gnn = ChebConv(hidden_dim, hidden_dim, K=cheb_K)
        self.act = nn.ReLU()

    def forward(
        self,
        h_seq: torch.Tensor,              # [P, N, H]
        edge_index: torch.Tensor,         # [2, E]
        edge_weight: Optional[torch.Tensor] = None,  # [E]
    ) -> torch.Tensor:
        P, N, H = h_seq.shape
        out_list = []
        for t in range(P):
            ht = h_seq[t]
            ht_spat = self.gnn(ht, edge_index, edge_weight=edge_weight)
            ht_spat = self.act(ht_spat)
            out_list.append(ht_spat)
        return torch.stack(out_list, dim=0)


# -------------------------------------------------
# 4) STGNN-Modell
# -------------------------------------------------

class STGNNWindowMultiRegressor(nn.Module):
    def __init__(
        self,
        num_nodes: int,
        embed_dim: int,
        hidden_dim: int,
        out_dim: int,
        in_dim: int = 1,
        gnn_type: str = "sage",
        temporal_type: str = "gru",
        spatial_pool: str = "mean",
        temp_hidden_dim: int = 32,
        temp_out_per_scale_week: int = 16,
        temp_out_per_scale_day: int = 16,
        time_in_dim: int = 0,
        cond_in_dim: int = 0,
        win_hour: int = 16,
        win_day: int = 64,
        win_week: int = 128,
        use_attention: bool = True,
        attn_heads: int = 4,  
        cheb_K: int = 3,
        length_encoder: int = 16,
    ):
        super().__init__()

        self.out_dim = out_dim
        self.hidden_dim = hidden_dim
        self.time_in_dim = time_in_dim
        self.cond_in_dim = cond_in_dim
        self.win_hour = win_hour
        self.win_day = win_day
        self.win_week = win_week

        self.emb = nn.Embedding(num_nodes, embed_dim)
        self.act = nn.ReLU()

        self.temp_enc = MultiScaleTemporalEncoder(
            in_dim=in_dim + embed_dim,
            hidden_dim=temp_hidden_dim,
            out_dim_per_scale_week=temp_out_per_scale_week,
            out_dim_per_scale_day=temp_out_per_scale_day,
            win_day=win_day,
            win_week=win_week,
            length_encoder=length_encoder, 
        )
        self.temp_out_dim = self.temp_enc.out_dim

        self.hour_proj = nn.Linear(in_dim + embed_dim, hidden_dim)
        self.spatial_enc = LocalSpatialEncoder(hidden_dim=hidden_dim, cheb_K=cheb_K)
        self.spatial_out_dim = hidden_dim

        self.core_st_dim = self.spatial_out_dim + self.temp_out_dim

        if time_in_dim > 0:
            self.time_mlp = nn.Sequential(
                nn.Linear(time_in_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, self.core_st_dim),
                nn.ReLU(),
            )
        else:
            self.time_mlp = None

        if cond_in_dim > 0:
            self.cond_mlp = nn.Sequential(
                nn.Linear(cond_in_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, self.core_st_dim),
                nn.ReLU(),
            )
        else:
            self.cond_mlp = None

        n_blocks = 1
        if self.time_mlp is not None:
            n_blocks += 1
        if self.cond_mlp is not None:
            n_blocks += 1

        self.st_proj = nn.Linear(n_blocks * self.core_st_dim, hidden_dim)

        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=False)
        
        # Self-attention pooling at the end
        self.use_attention = use_attention
        if self.use_attention:
            self.attn = nn.MultiheadAttention(
                embed_dim=hidden_dim,
                num_heads=attn_heads,
                batch_first=False, 
            )
            # global query vector that learns how to aggregate over time
            self.global_query = nn.Parameter(
                torch.randn(1, hidden_dim)
            )


        self.head = nn.Linear(hidden_dim, out_dim)

    def forward(
        self,
        x_window: torch.Tensor,              # [W, N, F]
        edge_index: torch.Tensor,            # [2, E]
        edge_weight: Optional[torch.Tensor] = None, 
        time_feat: Optional[torch.Tensor] = None,  # [W, time_in_dim]
        condat: Optional[torch.Tensor] = None,     # [N, cond_in_dim]
    ) -> torch.Tensor:

        W, N, F = x_window.shape
        device = x_window.device

        node_emb = self.emb.weight.to(device)                        # [N, embed_dim]
        node_emb_exp = node_emb.unsqueeze(0).expand(W, -1, -1)       # [W, N, embed_dim]
        x_with_emb = torch.cat([x_window, node_emb_exp], dim=-1)     # [W, N, F+embed_dim]

        if self.win_hour > W:
            raise ValueError(f"win_hour={self.win_hour} > Window Size W={W}")
        x_hour = x_with_emb[-self.win_hour:]                         # [Wh, N, F+emb]

        h_hour = self.hour_proj(x_hour)                              # [Wh, N, hidden_dim]
        h_hour = self.act(h_hour)

        h_spat_hour = self.spatial_enc(h_hour, edge_index, edge_weight=edge_weight)   # [Wh, N, hidden_dim]

        st_tw = self.temp_enc(x_with_emb)                            # [P_tw, N, temp_out_dim]

        P = min(h_spat_hour.size(0), st_tw.size(0))
        h_spat_hour = h_spat_hour[-P:]                               # [P, N, hidden_dim]
        st_tw       = st_tw[-P:]                                     # [P, N, temp_out_dim]

        st_core = torch.cat([h_spat_hour, st_tw], dim=-1)            # [P, N, core_st_dim]
        blocks = [st_core]

        if self.time_mlp is not None:
            if time_feat is None:
                raise ValueError("time_feat cannot be none if time_in_dim > 0.")

            if time_feat.size(0) < P:
                raise ValueError(f"time_feat too short: {time_feat.size(0)} < P={P}")

            time_feat = time_feat[-P:]             # [P, time_in_dim]

            t_emb = self.time_mlp(time_feat.to(device))   # [P, core_st_dim]
            t_emb = t_emb.unsqueeze(1).expand(-1, N, -1)  # [P, N, core_st_dim]
            blocks.append(t_emb)

        if self.cond_mlp is not None:
            if condat is None:
                raise ValueError("condat cannot be none if cond_in_dim > 0.")
            if condat.size(0) != N:
                raise ValueError(f"condat expects N={N}, receives {condat.size(0)}.")
            c_emb = self.cond_mlp(condat.to(device))                 # [N, core_st_dim]
            c_emb = c_emb.unsqueeze(0).expand(P, -1, -1)             # [P, N, core_st_dim]
            blocks.append(c_emb)

        st_cat = torch.cat(blocks, dim=-1)                           # [P, N, n_blocks * core_st_dim]
        h_seq = self.st_proj(st_cat)                                 # [P, N, hidden_dim]
        h_seq = self.act(h_seq)

        out_seq, h_n = self.gru(h_seq)          # out_seq: [P, N, H], h_n: [1, N, H]

        if self.use_attention:
            # out_seq: [P, N, H] -> Keys und Values
            # Query: [1, N, H]
            P, N, H = out_seq.shape
            device = out_seq.device

            q = self.global_query.unsqueeze(1).expand(-1, N, -1)   # [1, N, H]
            # MultiheadAttention erwartet [S, B, E]
            attn_out, attn_weights = self.attn(
                q,              # query: [1, N, H]
                out_seq,        # key:   [P, N, H]
                out_seq,        # value: [P, N, H]
            )
            # attn_out: [1, N, H]
            h_final = attn_out[0]         # [N, H]
        else:
            # Fallback:
            h_final = h_n[-1]             # [N, H]

        out = self.head(h_final)          # [N, out_dim]
        return out



# =============================
# e) Training and evaluation (multi-step, multi-target)
# =============================

@dataclass
class STTrainConfig:
    lr: float = LR
    weight_decay: float = WEIGHT_DECAY
    epochs: int = EPOCHS
    patience: int = PATIENCE
    window_size: int = WINDOW_SIZE
    horizon_steps: int = HORIZON_STEPS  # per Target

    # Modell-Hyperparameter (tunable)
    embed_dim: int = EMBED_DIM
    hidden_dim: int = HIDDEN_DIM
    temp_hidden_dim: int = 32
    temp_out_per_scale_week: int = 16
    temp_out_per_scale_day: int = 16
    win_hour: int = 16
    win_day: int = 64
    win_week: int = 128
    cheb_K: int = 3
    length_encoder: int = LENGTH_ENCODER_DEFAULT


def _compute_window_metrics_multi_per_horizon(
    model: nn.Module,
    data: STWindowDataMulti,
    end_indices: np.ndarray,
    device: torch.device,
    loss_nodes: torch.Tensor,
    window_size: int,
    eval_horizons: List[int],
    time_full: torch.Tensor,
) -> Tuple[dict, float]:

    x_full = data.x.to(device)
    y_full = data.y.to(device)
    edge_index = data.edge_index.to(device)
    edge_weight = data.edge_weight.to(device)
    time_full = time_full.to(device)
    W = window_size

    out_dim = y_full.shape[2]      # H * M
    M = len(data.target_cols)
    H_tot = out_dim // M

    sum_abs = {h: 0.0 for h in eval_horizons}
    sum_sq  = {h: 0.0 for h in eval_horizons}
    count   = {h: 0   for h in eval_horizons}

    model.eval()
    with torch.no_grad():
        for end_t in end_indices:
            start_t = end_t - W + 1
            x_win = x_full[start_t:end_t+1]        # [W, N, F]
            y_target = y_full[end_t]               # [N, H*M]

            time_win = time_full[start_t:end_t+1]  # [W, time_dim]

            pred = model(
                x_win,
                edge_index,
                edge_weight=edge_weight,
                time_feat=time_win,
                condat=None,
            )

            y_nodes = y_target[loss_nodes]         # [N_sel, H*M]
            p_nodes = pred[loss_nodes]             # [N_sel, H*M]

            if not torch.isfinite(y_nodes).any():
                continue

            N_sel = y_nodes.shape[0]

            y_nodes_resh = y_nodes.view(N_sel, M, H_tot)
            p_nodes_resh = p_nodes.view(N_sel, M, H_tot)

            for h_step in eval_horizons:
                h_idx = h_step - 1
                if h_idx < 0 or h_idx >= H_tot:
                    continue

                y_h = y_nodes_resh[..., h_idx]     # [N_sel, M]
                p_h = p_nodes_resh[..., h_idx]     # [N_sel, M]

                mask_h = torch.isfinite(y_h)
                if mask_h.sum() == 0:
                    continue

                y_valid = y_h[mask_h]
                p_valid = p_h[mask_h]

                diff = p_valid - y_valid

                sum_abs[h_step] += diff.abs().sum().item()
                sum_sq[h_step]  += (diff ** 2).sum().item()
                count[h_step]   += mask_h.sum().item()

    metrics_per_h = {}
    mae_values_for_agg = []

    for h_step in eval_horizons:
        if count[h_step] == 0:
            metrics_per_h[h_step] = {"MAE": float("nan"), "RMSE": float("nan")}
            continue

        mae_h = sum_abs[h_step] / count[h_step]
        rmse_h = math.sqrt(sum_sq[h_step] / count[h_step])

        mae_values_for_agg.append(mae_h)
        metrics_per_h[h_step] = {"MAE": float(mae_h), "RMSE": float(rmse_h)}

    if len(mae_values_for_agg) == 0:
        agg_mae = float("nan")
    else:
        agg_mae = float(np.mean(mae_values_for_agg))

    return metrics_per_h, agg_mae


def train_stgnn_window_multi(
    data: STWindowDataMulti,
    cfg: STTrainConfig,
    gnn_type: str = "sage",
    temporal_type: str = "gru",
    eval_horizons: Optional[List[int]] = None,
) -> dict:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    x_full = data.x.to(device)
    y_full = data.y.to(device)
    edge_index = data.edge_index.to(device)
    edge_weight = data.edge_weight.to(device)
    W = cfg.window_size
    
    time_full = torch.tensor(data.time_feat, dtype=torch.float32, device=device)
    time_in_dim = time_full.shape[1]

    M = len(data.target_cols)
    H_per_target = cfg.horizon_steps
    out_dim = H_per_target * M
    in_dim = x_full.shape[2]

    if eval_horizons is None:
        eval_horizons = [1, 2, 4]

    model = STGNNWindowMultiRegressor(
        num_nodes=x_full.shape[1],
        embed_dim=cfg.embed_dim,
        hidden_dim=cfg.hidden_dim,
        out_dim=out_dim,
        in_dim=in_dim,
        gnn_type=gnn_type,
        temporal_type=temporal_type,
        time_in_dim=time_in_dim,
        temp_hidden_dim=cfg.temp_hidden_dim,
        temp_out_per_scale_week=cfg.temp_out_per_scale_week,
        temp_out_per_scale_day=cfg.temp_out_per_scale_day,
        win_hour=cfg.win_hour,
        win_day=cfg.win_day,
        win_week=cfg.win_week,
        cheb_K=cfg.cheb_K,
        length_encoder=cfg.length_encoder, 
    ).to(device)

    opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = nn.SmoothL1Loss()

    loss_nodes = torch.tensor(data.valid_nodes, dtype=torch.long, device=device)

    best_state = None
    best_val = float("inf")
    wait = 0

    for epoch in range(cfg.epochs):
        model.train()
        epoch_loss = 0.0
        count_windows = 0

        for end_t in data.train_end_idx:
            start_t = end_t - W + 1

            x_win = x_full[start_t:end_t+1]
            y_target = y_full[end_t]

            time_win = time_full[start_t:end_t+1]

            pred = model(
                x_win,
                edge_index,
                edge_weight=edge_weight,
                time_feat=time_win,
                condat=None,
            )

            y_nodes = y_target[loss_nodes]
            p_nodes = pred[loss_nodes]

            mask = torch.isfinite(y_nodes)
            if mask.sum() == 0:
                continue

            loss = loss_fn(p_nodes[mask], y_nodes[mask])

            opt.zero_grad()
            loss.backward()
            opt.step()

            epoch_loss += loss.item()
            count_windows += 1

        avg_train_loss = epoch_loss / max(count_windows, 1)

        val_per_h, val_agg_mae = _compute_window_metrics_multi_per_horizon(
            model, data, data.val_end_idx, device, loss_nodes, cfg.window_size,
            eval_horizons, time_full
        )

        if val_agg_mae < best_val:
            best_val = val_agg_mae
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= cfg.patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_per_h, _ = _compute_window_metrics_multi_per_horizon(
        model, data, data.val_end_idx, device, loss_nodes, cfg.window_size,
        eval_horizons, time_full
    )
    test_per_h, _ = _compute_window_metrics_multi_per_horizon(
        model, data, data.test_end_idx, device, loss_nodes, cfg.window_size,
        eval_horizons, time_full
    )

    return {
        "model": model,
        "metrics": {
            "val_per_h": val_per_h,
            "test_per_h": test_per_h,
        },
        "val_agg_mae": float(best_val),
    }


# =============================
# f) High-level API
# =============================

def train_stgnn_window_from_df(
    df: pd.DataFrame,
    *,
    seed: int = SEED,
    window_size: int = WINDOW_SIZE,
    horizon_steps: int = HORIZON_STEPS,
    feature_cols: Optional[List[str]] = None,
    target_cols: Optional[List[str]] = None,
    stride: int = 1,
    eval_horizons: Optional[List[int]] = None,
    hp_overrides: Optional[Dict[str, Any]] = None, 
) -> dict:

    set_seed(seed)
    if eval_horizons is None:
        eval_horizons = [1, 2, 4]

    data = make_window_data_multi_from_df(
        df,
        window_size=window_size,
        horizon_steps=horizon_steps,
        feature_cols=feature_cols,
        target_cols=target_cols,
        stride=stride,
    )

    cfg = STTrainConfig(
        window_size=window_size,
        horizon_steps=horizon_steps,
    )

    if hp_overrides is not None:
        for k, v in hp_overrides.items():
            if hasattr(cfg, k):
                setattr(cfg, k, v)
            else:
                raise ValueError(f"Unknown Hyperparameters in hp_overrides: {k}")

    result = train_stgnn_window_multi(
        data,
        cfg,
        eval_horizons=eval_horizons,
    )

    return {
        "model": result["model"],
        "metrics": result["metrics"],
        "data": data,
        "window_size": window_size,
        "horizon_steps": horizon_steps,
        "feature_cols": data.feature_cols,
        "target_cols": data.target_cols,
    }


# =============================
# g) Multi-Seed Experiments
# =============================

def run_multi_seed_experiment(
    df: pd.DataFrame,
    *,
    seeds: List[int],
    window_size: int = WINDOW_SIZE,
    horizon_steps: int = HORIZON_STEPS,
    feature_cols: Optional[List[str]] = None,
    target_cols: Optional[List[str]] = None,
    stride: int = 1,
    eval_horizons: Optional[List[int]] = None,
    hp_overrides: Optional[Dict[str, Any]] = None,   # <--- neu
) -> dict:

    if eval_horizons is None:
        eval_horizons = [1, 2, 4]

    all_results = []

    metrics_collect = {
        "val_per_h": {h: {"MAE": [], "RMSE": []} for h in eval_horizons},
        "test_per_h": {h: {"MAE": [], "RMSE": []} for h in eval_horizons},
    }

    for s in seeds:
        res = train_stgnn_window_from_df(
            df,
            seed=s,
            window_size=window_size,
            horizon_steps=horizon_steps,
            feature_cols=feature_cols,
            target_cols=target_cols,
            stride=stride,
            eval_horizons=eval_horizons,
            hp_overrides=hp_overrides,
        )
        all_results.append(res)

        for split_key in ["val_per_h", "test_per_h"]:
            for h in eval_horizons:
                for metric in ["MAE", "RMSE"]:
                    metrics_collect[split_key][h][metric].append(
                        res["metrics"][split_key][h][metric]
                    )

    metrics_stats = {
        "val_per_h": {},
        "test_per_h": {},
    }

    for split_key in ["val_per_h", "test_per_h"]:
        for h in eval_horizons:
            metrics_stats[split_key][h] = {}
            for metric in ["MAE", "RMSE"]:
                values = np.array(metrics_collect[split_key][h][metric], dtype=float)
                metrics_stats[split_key][h][metric + "_mean"] = float(values.mean())
                metrics_stats[split_key][h][metric + "_var"] = float(values.var(ddof=1))

    return {
        "seeds": seeds,
        "all_results": all_results,
        "metrics_raw": metrics_collect,
        "metrics_stats": metrics_stats,
    }


# =============================
# h) Hyperparameter Tuning
# =============================

def sample_hyperparams(
    rng: np.random.Generator,
) -> Dict[str, Any]:
    embed_dim_choices = [8, 16, 32, 64]
    hidden_dim_choices = [32, 64, 128, 256]
    temp_hidden_choices = [32, 64, 128]
    temp_out_choices_week = [16, 32, 64, 128]
    temp_out_choices_day = [16, 32, 64]

    win_hour_choices = [4]
    win_day_choices = [96]
    win_week_choices = [672]
    
    cheb_K_choices = [2, 3, 4, 5]
    length_encoder_choices = [16]

    def log_uniform(low, high):
        log_low, log_high = np.log10(low), np.log10(high)
        val = rng.uniform(log_low, log_high)
        return float(10 ** val)

    hp = {
        "lr": log_uniform(1e-5, 5e-3),
        "weight_decay": log_uniform(1e-6, 1e-3),
        "embed_dim": int(rng.choice(embed_dim_choices)),
        "hidden_dim": int(rng.choice(hidden_dim_choices)),
        "temp_hidden_dim": int(rng.choice(temp_hidden_choices)),
        "temp_out_per_scale_week": int(rng.choice(temp_out_choices_week)),
        "temp_out_per_scale_day": int(rng.choice(temp_out_choices_day)),
        "win_hour": int(rng.choice(win_hour_choices)),
        "win_day": int(rng.choice(win_day_choices)),
        "win_week": int(rng.choice(win_week_choices)),
        "cheb_K": int(rng.choice(cheb_K_choices)),
        "length_encoder": int(rng.choice(length_encoder_choices)),
    }
    return hp


@dataclass(frozen=True)
class TrialResult:
    trial: int
    hyperparams: Dict[str, Any]
    score: float
    metrics_stats: Dict[str, Any]
    result: Dict[str, Any]


def _run_one_trial(
    trial: int,
    df: pd.DataFrame,
    *,
    seeds: List[int],
    base_seed: int,
    window_size: int,
    horizon_steps: int,
    feature_cols: Optional[List[str]],
    target_cols: Optional[List[str]],
    stride: int,
    eval_horizons: List[int],
) -> TrialResult:
    # deterministischer RNG pro Trial
    rng = np.random.default_rng(base_seed + trial)

    hp = sample_hyperparams(rng)

    res = run_multi_seed_experiment(
        df,
        seeds=seeds,
        window_size=window_size,
        horizon_steps=horizon_steps,
        feature_cols=feature_cols,
        target_cols=target_cols,
        stride=stride,
        eval_horizons=eval_horizons,
        hp_overrides=hp,
    )

    stats = res["metrics_stats"]
    mae_list = [stats["val_per_h"][h]["MAE_mean"] for h in eval_horizons]
    score = float(np.mean(mae_list))

    return TrialResult(
        trial=trial,
        hyperparams=hp,
        score=score,
        metrics_stats=stats,
        result=res,
    )


def random_search_multi_seed_parallel(
    df: pd.DataFrame,
    *,
    seeds: List[int],
    n_trials: int = 20,
    base_seed: int = 42,
    window_size: int = WINDOW_SIZE,
    horizon_steps: int = HORIZON_STEPS,
    feature_cols: Optional[List[str]] = None,
    target_cols: Optional[List[str]] = None,
    stride: int = 1,
    eval_horizons: Optional[List[int]] = None,
    max_workers: Optional[int] = None,
) -> dict:
    if eval_horizons is None:
        eval_horizons = [1, 2, 4]

    best_score = float("inf")
    best_hp = None
    best_res = None
    trials = []

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = [
            ex.submit(
                _run_one_trial,
                t,
                df,
                seeds=seeds,
                base_seed=base_seed,
                window_size=window_size,
                horizon_steps=horizon_steps,
                feature_cols=feature_cols,
                target_cols=target_cols,
                stride=stride,
                eval_horizons=eval_horizons,
            )
            for t in range(n_trials)
        ]

        done = 0
        for fut in as_completed(futures):
            tr = fut.result()
            done += 1

            trials.append({
                "trial": tr.trial,
                "hyperparams": tr.hyperparams,
                "score": tr.score,
                "metrics_stats": tr.metrics_stats,
            })

            if tr.score < best_score:
                best_score = tr.score
                best_hp = tr.hyperparams
                best_res = tr.result

            print(f"[RandomSearch] Done {done}/{n_trials}: "
                  f"trial={tr.trial}, score={tr.score:.4f}, best={best_score:.4f}")

    trials.sort(key=lambda x: x["trial"])

    return {
        "best_score": best_score,
        "best_hyperparams": best_hp,
        "best_result": best_res,
        "trials": trials,
    }

In [6]:
rs = random_search_multi_seed_parallel(
    df,
    seeds=[0],
    n_trials=20,
    feature_cols=["e5"],
    target_cols=["e5"],
)

print(rs["best_hyperparams"], rs["best_score"])

KeyboardInterrupt: 